# LangSmith: Tracing, Datasets & Experiments

In [ ]:
%pip install langsmith openai openevals datasets python-dotenv -q

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import json
from dotenv import load_dotenv

load_dotenv()  # loads LANGSMITH_API_KEY, OPENAI_API_KEY, ANTHROPIC_API_KEY from .env

# LangSmith EU endpoint — set here so it takes effect before the client initialises
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"]  = "content-creation-blog-title-eval"

from langsmith import Client
from langsmith.wrappers import wrap_openai
from langsmith import traceable
from openai import OpenAI
from openevals.llm import create_llm_as_judge
from datasets import load_dataset

client = Client()
openai_client = wrap_openai(OpenAI())

/home/bella/miniforge3/envs/consulting/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Tracing

`@traceable` logs every call to LangSmith automatically. `wrap_openai` adds tracing to the OpenAI client.

In [ ]:
@traceable(name="simple-qa")
def answer_question(question: str) -> str:
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Answer concisely."},
            {"role": "user",   "content": question},
        ],
    )
    return response.choices[0].message.content.strip()

# This call will appear in your LangSmith project under 'content-creation-blog-title-eval'
answer = answer_question("What is the speed of light?")
print(answer)

The speed of light in a vacuum is approximately 299,792,458 meters per second (about 300,000 kilometers per second or 186,282 miles per second).


## 2. Custom dataset: blog title generation (Content Creation domain)

Our project's domain is **Content Creation**: given a real Medium article excerpt, generate an engaging, accurate title. Examples were extracted from [`fabiochiu/medium-articles`](https://huggingface.co/datasets/fabiochiu/medium-articles) by `scripts/extract_examples.py` and saved to `data/blog_title_examples.json` (see `README.md`'s "Data Structure" section for the extraction/quality-filtering details, including a title-leakage guard since Medium's scraped text often repeats the headline as the first line of the body).

Note: the original version of this notebook loaded `cais/hle` (Humanity's Last Exam) here as a generic multiple-choice benchmark. That dataset is **gated** on the Hugging Face Hub and requires authentication, which isn't relevant to our project anyway — we use our own domain-specific dataset instead.

In [ ]:
# Load the structured blog-title-generation examples (see scripts/extract_examples.py)
with open("data/blog_title_examples.json") as f:
    blog_examples = json.load(f)

print(f"Loaded {len(blog_examples)} examples")
tag_counts = {}
for ex in blog_examples:
    tag_counts[ex["metadata"]["primary_tag"]] = tag_counts.get(ex["metadata"]["primary_tag"], 0) + 1
print("Primary tag distribution:", tag_counts)

Loaded 18 examples
Primary tag distribution: {'Mental Health': 2, 'Brain': 1, 'Society': 1, 'Books': 1, 'People': 1, 'Self Improvement': 1, 'Entrepreneurship': 1, 'Productivity': 2, 'Health': 2, 'Writing': 1, 'Lifestyle': 1, 'Work': 1, 'Marketing': 2, 'Creativity': 1}


In [ ]:
# Push to a LangSmith dataset (see scripts/create_dataset.py for the full script)
DATASET_NAME = "blog-title-generation-medium-articles"
DATASET_DESCRIPTION = (
    "Blog title generation task: given a ~400-word excerpt of a real Medium article, "
    "generate an engaging, accurate title. Ground truth is the article's actual "
    "published title. 18 examples spanning 14 topics, sourced from fabiochiu/medium-articles."
)

# Create dataset (skip if it already exists)
existing = {d.name: d for d in client.list_datasets()}
if DATASET_NAME not in existing:
    dataset = client.create_dataset(DATASET_NAME, description=DATASET_DESCRIPTION)
    client.create_examples(
        inputs     = [ex["inputs"]   for ex in blog_examples],
        outputs    = [ex["outputs"]  for ex in blog_examples],
        metadata   = [ex["metadata"] for ex in blog_examples],
        dataset_id = dataset.id,
    )
    print(f"Created dataset '{DATASET_NAME}' with {len(blog_examples)} examples")
else:
    dataset = existing[DATASET_NAME]
    print(f"Dataset '{DATASET_NAME}' already exists (id={dataset.id}) — skipping upload")

Created dataset 'blog-title-generation-medium-articles' with 18 examples


> **Note:** Sections 3–5 above (target function, evaluator, experiment run) are now fully adapted to the blog-title-generation task and have been run for real - see `data/eval_results_gpt-4o-mini.json` for the collected per-example results (Step 9 analysis). The "Compare two models (A/B)" and "Reproducibility" sections below still show the original multiple-choice example; those map to the lab's optional Step 14 (cost/performance A/B comparison) and the extension activities, not a required step.

## 3. Target function: blog title generation

A target function maps dataset inputs → outputs. It's decorated with `@traceable` so every call is automatically logged to LangSmith as a run, and it calls the model through the already-wrapped `openai_client` so the underlying LLM call is traced as a nested child run.

See `scripts/target_function.py` for the standalone version.

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert blog editor. Given the excerpt of a blog article, write a single, "
    "engaging, accurate title for it - the kind a human writer would actually publish on "
    "Medium. Reply with ONLY the title text: no quotes, no explanation, no \"Title:\" prefix."
)

@traceable(name="blog-title-target")
def generate_title(inputs: dict) -> dict:
    """inputs: {"article_text": str} -> outputs: {"title": str}"""
    article_text = inputs["article_text"]
    try:
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": article_text},
            ],
            temperature=0.3,
            max_tokens=40,
        )
        title = response.choices[0].message.content.strip().strip('"')
        return {"title": title}
    except Exception as e:
        return {"title": "", "error": str(e)}

# Smoke test on the first 3 examples
for ex in blog_examples[:3]:
    result = generate_title(ex["inputs"])
    print("REFERENCE:", ex["outputs"]["title"])
    print("GENERATED:", result["title"])
    print("-" * 60)

REFERENCE: Mental Note Vol. 24
GENERATED: Celebrating Connection: Our Top Stories on Mental Health and Resilience This Holiday Season
------------------------------------------------------------
REFERENCE: Your Brain On Coronavirus
GENERATED: The Hidden Toll of COVID-19: Understanding the Long-Term Impact on Health and Isolation
------------------------------------------------------------
REFERENCE: Surviving a Rod Through the Head
GENERATED: The Remarkable Case of Phineas Gage: How a Brain Injury Redefined Neuroscience and Personality
------------------------------------------------------------


In [ ]:
# Evaluator: custom correctness rubric for blog titles, judged by gpt-4o-mini.
#
# We don't use openevals' built-in CORRECTNESS_PROMPT here: it's written for factual
# QA ("addresses all parts of the question", "factual errors") which doesn't map cleanly
# onto judging a generated title against a reference title - there's rarely one single
# "correct" title, so we want to reward topical/factual alignment with the article and
# reference, not penalize a title just for being phrased differently.
#
# See scripts/evaluator.py for the standalone version.
BLOG_TITLE_CORRECTNESS_PROMPT = """You are an expert blog editor evaluating whether a generated title \
is a correct, publishable title for the given article excerpt.

<Rubric>
A correct title:
- Accurately represents the article's actual topic and content (no factual mismatch)
- Would be a reasonable title a human editor could publish for this article
- Captures the same core subject as the reference title, even if the wording differs

Note: there is rarely one single "correct" title. Do NOT penalize a title just for using \
different words, a different structure, or a different length than the reference - only \
penalize it for being topically inaccurate, misleading, or unrelated to what the article \
actually discusses.

When scoring, you should penalize:
- Titles that misrepresent or contradict the article's actual content
- Titles that are generic to the point of being meaningless (e.g. "An Article About Life")
- Titles that are unrelated to the reference title's subject matter
</Rubric>

<Instructions>
- Read the article excerpt in <input>, the generated title in <output>, and the reference \
title in <reference_outputs>
- Judge topical/factual alignment with the article and reference - not stylistic similarity
</Instructions>

<input>
{inputs}
</input>

<output>
{outputs}
</output>

<reference_outputs>
{reference_outputs}
</reference_outputs>"""

_judge = create_llm_as_judge(
    prompt=BLOG_TITLE_CORRECTNESS_PROMPT,
    model="openai:gpt-4o-mini",
    feedback_key="correctness",
    continuous=True,
)

def correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    return _judge(inputs=inputs, outputs=outputs, reference_outputs=reference_outputs)

# Smoke test on one example
ex = blog_examples[0]
generated = generate_title(ex["inputs"])
result = correctness_evaluator(inputs=ex["inputs"], outputs=generated, reference_outputs=ex["outputs"])
print("REFERENCE:", ex["outputs"]["title"])
print("GENERATED:", generated["title"])
print("SCORE:", result["score"])
print("REASONING:", result["comment"])

REFERENCE: Mental Note Vol. 24
GENERATED: Celebrating Connection: Our Top Stories on Mental Health and Resilience This Holiday Season
SCORE: 0.9
REASONING: The generated title accurately represents the article's content, which discusses appreciation for readers, the importance of mental health, and shares curated top stories related to these themes during the holiday season. It captures the core subject matter of the article very well, though it expands on some elements the reference title does not address. The minor discrepancy lies in the promotional tone, which is not present in the reference title, but it isn't misleading or inaccurate.


In [ ]:
# Step 7: run the evaluation experiment against the full dataset.
# See scripts/run_evaluation.py for the standalone version (also exports
# per-example results to data/eval_results_gpt-4o-mini.json for Step 9 analysis).
results = client.evaluate(
    generate_title,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o-mini-blog-title",
    max_concurrency=3,
)
print(results)

View the evaluation results for experiment: 'gpt-4o-mini-blog-title-41b50506' at:
https://eu.smith.langchain.com/o/80bd18ed-fa6b-4055-96c9-9b77785a8b50/datasets/c55c213c-5c11-48e2-9bbd-04c91b31ba7d/compare?selectedSessions=20a00854-8185-4053-a9fb-202f72a2f0c6

<ExperimentResults gpt-4o-mini-blog-title-41b50506>


## 4. Compare two models (A/B)

Run the same benchmark with a different model. The LangSmith UI lets you compare both experiments side-by-side.

In [ ]:
def target_gpt4o(inputs: dict) -> dict:
    response = openai_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are answering a multiple-choice exam question. Reply with only the letter of the correct answer (A, B, C, or D)."},
            {"role": "user",   "content": inputs["question"]},
        ],
        max_tokens=10,
    )
    return {"answer": response.choices[0].message.content.strip()}

results_4o = client.evaluate(
    target_gpt4o,
    data=DATASET_NAME,
    evaluators=[correctness_evaluator],
    experiment_prefix="gpt-4o",
    max_concurrency=2,
)
print(results_4o)

View the evaluation results for experiment: 'gpt-4o-d6ccf9cc' at:
https://eu.smith.langchain.com/o/87335986-1828-40f0-9dbd-323bf1524826/datasets/7d72e56b-14f4-4c09-bd15-2f66f6d0dd12/compare?selectedSessions=41943530-8586-44bc-807a-bc89640c015e


<ExperimentResults gpt-4o-d6ccf9cc>


In [ ]:
import pandas as pd

scores_mini = [r["feedback"]["correctness"] for r in results_mini._results if "correctness" in r.get("feedback", {})]
scores_4o   = [r["feedback"]["correctness"] for r in results_4o._results   if "correctness" in r.get("feedback", {})]

summary = pd.DataFrame({
    "model":     ["gpt-4o-mini", "gpt-4o"],
    "mean_score": [pd.Series(scores_mini).mean(), pd.Series(scores_4o).mean()],
    "n":         [len(scores_mini), len(scores_4o)],
})
print(summary)
print("\n→ Are these scores actually different? See notebook 03 for the statistical test.")

         model  mean_score  n
0  gpt-4o-mini         NaN  0
1       gpt-4o         NaN  0

→ Are these scores actually different? See notebook 03 for the statistical test.


## 5. Reproducibility

From the slides: same code + seed ≠ guaranteed identical results. Even small implementation differences can shift scores.

In [ ]:
# Run the same single example twice — do you always get the same answer?
question = hle_examples[0]["question"]

run1 = target_gpt4o_mini({"question": question})
run2 = target_gpt4o_mini({"question": question})

print("Run 1:", run1["answer"])
print("Run 2:", run2["answer"])
print("Identical:", run1["answer"] == run2["answer"])
print("\nAt temperature=0 this is usually stable — but never guaranteed across API versions.")

Run 1: C
Run 2: C. Non-Elitism
Identical: False

At temperature=0 this is usually stable — but never guaranteed across API versions.
